# Guia de configuracao do servidor e validacao passo a passo

Este notebook explica, de forma isolada e didatica, como configurar cada peca da biblioteca `load_balancer_sim` antes de rodar os experimentos completos:

1. `SimulationConfig` - os parametros de uma rodada;
2. `Server` - capacidade e tempo de servico de cada servidor;
3. `LoadBalancer` - como uma requisicao escolhe um servidor;
4. `PoissonTrafficGenerator` - como as chegadas sao geradas;
5. Uma simulacao curta com **rastro visivel de cada evento**, para voce conferir visualmente se tudo esta funcionando;
6. Uma verificacao automatica das invariantes do sistema (`invariants.py`), que confirma matematicamente que nao houve inconsistencia.

A ideia é que, depois de rodar este notebook, voce tenha confianca de que os numeros produzidos em [experiments.ipynb](./experiments.ipynb) fazem sentido.

In [11]:
from __future__ import annotations

import simpy

from load_balancer_sim.config import SimulationConfig
from load_balancer_sim.server import Server
from load_balancer_sim.load_balancer import LoadBalancer
from load_balancer_sim.traffic import PoissonTrafficGenerator, generate_poisson_arrival_times
from load_balancer_sim.metrics import MetricsCollector
from load_balancer_sim.invariants import validate_simulation_invariants, SimulationInvariantError

## 1) `SimulationConfig`: os parametros de uma rodada

Cada campo controla um aspecto da simulacao:

| Campo | Significado |
| --- | --- |
| `policy` | politica de roteamento: `random`, `round_robin` ou `shortest_queue` |
| `server_count` | quantidade de servidores homogeneos |
| `server_capacity` | quantas requisicoes cada servidor processa ao mesmo tempo |
| `service_time` | tempo (determinístico) gasto para atender uma requisicao |
| `arrival_rate` | taxa `lambda` do processo de chegada Poisson |
| `horizon` | duracao maxima da simulacao |
| `seed` | semente para reproducibilidade (roteamento e, se aplicavel, tracos aleatorios) |

A classe valida os valores automaticamente: por exemplo, `arrival_rate` e `service_time` precisam ser numeros finitos e positivos, e `policy` precisa ser uma das tres suportadas.

In [15]:
config = SimulationConfig(
    policy="random",
    server_count=3,
    server_capacity=1,
    service_time=1.0,
    arrival_rate=1.5,
    horizon=5.0,
    seed=7,
)

print("Configuracao validada com sucesso:")
for field_name in config.__dataclass_fields__:
    print(f"  {field_name:15s} = {getattr(config, field_name)}")

Configuracao validada com sucesso:
  policy          = random
  server_count    = 3
  server_capacity = 1
  service_time    = 1.0
  arrival_rate    = 1.5
  horizon         = 5.0
  seed            = 7


In [16]:
# Teste de validacao: um valor invalido deve ser rejeitado com uma mensagem clara.
try:
    SimulationConfig(arrival_rate=-1.0)
except (TypeError, ValueError) as exc:
    print(f"Erro esperado ao usar arrival_rate invalido: {exc}")

Erro esperado ao usar arrival_rate invalido: arrival_rate deve ser positivo


## 2) `Server`: capacidade e tempo de servico

Cada `Server` usa um `simpy.Resource` com `capacity` slots. Requisicoes acima da capacidade esperam em uma fila FIFO. `service_time` é o tempo gasto processando cada requisicao (precisa ser modificado para que seja exponencial).

Abaixo criamos os servidores exatamente com os valores de `config` e inspecionamos o estado inicial de cada um.

In [17]:
environment = simpy.Environment()

servers = [
    Server(
        environment,
        server_id=index,
        capacity=config.server_capacity,
        service_time=config.service_time,
    )
    for index in range(config.server_count)
]

for server in servers:
    print(
        f"Servidor {server.id}: capacidade={server.capacity}, "
        f"service_time={server.service_time}, "
        f"active_count={server.active_count}, waiting_count={server.waiting_count}"
    )

Servidor 0: capacidade=1, service_time=1.0, active_count=0, waiting_count=0
Servidor 1: capacidade=1, service_time=1.0, active_count=0, waiting_count=0
Servidor 2: capacidade=1, service_time=1.0, active_count=0, waiting_count=0


## 3) `LoadBalancer`: como uma requisicao escolhe um servidor

O `LoadBalancer` recebe a lista de servidores e delega a escolha para uma politica:

- `random`: escolhe um servidor aleatorio (semente de `simulation_config.seed`);
- `round_robin`: alterna ciclicamente entre os servidores;
- `shortest_queue`: escolhe o servidor com menor `active_count + waiting_count`.

Para deixar isso visivel, vamos rotear manualmente algumas requisicoes falsas (sem avancar o relogio) e observar qual servidor cada uma recebe.

In [20]:
from load_balancer_sim.request import Request

demo_env = simpy.Environment()
demo_servers = [
    Server(demo_env, server_id=index, capacity=config.server_capacity, service_time=config.service_time)
    for index in range(config.server_count)
]
demo_load_balancer = LoadBalancer(
    demo_env,
    demo_servers,
    policy=config.policy,
    simulation_config=config,
)

print(f"Politica em uso: {config.policy}\n")
for request_id in range(6):
    demo_request = Request(id=request_id, burst_id=0, arrival_time=float(demo_env.now))
    chosen_server = demo_load_balancer.route_request(demo_request)
    # Avancamos o relogio um instante infinitesimal para que o processo do
    # SimPy realmente execute e o slot seja ocupado antes de imprimirmos o
    # estado. Sem isso, active_count/waiting_count ficariam sempre em 0 aqui,
    # pois o processo agendado ainda nao teria rodado nenhum passo.
    demo_env.run(until=demo_env.now + 1e-6)
    print(
        f"Requisicao {request_id:>2} -> servidor {chosen_server.id} "
        f"(active={chosen_server.active_count}, waiting={chosen_server.waiting_count})"
    )

demo_env.run()

Politica em uso: random

Requisicao  0 -> servidor 1 (active=1, waiting=0)
Requisicao  1 -> servidor 0 (active=1, waiting=0)
Requisicao  2 -> servidor 1 (active=1, waiting=1)
Requisicao  3 -> servidor 2 (active=1, waiting=0)
Requisicao  4 -> servidor 0 (active=1, waiting=1)
Requisicao  5 -> servidor 0 (active=1, waiting=2)


## 4) `PoissonTrafficGenerator`: como as chegadas sao geradas

Dado `arrival_rate` (lambda), os intervalos entre chegadas sao exponenciais com media `1/lambda`. Abaixo geramos os instantes de chegada para o `horizon` da configuracao e imprimimos cada um, junto com o intervalo em relacao a chegada anterior — assim voce pode conferir visualmente que os intervalos parecem exponenciais e que nenhuma chegada ultrapassa o horizonte.

In [7]:
arrival_times = generate_poisson_arrival_times(
    arrival_rate=config.arrival_rate,
    horizon=config.horizon,
    seed=config.seed,
)

print(f"lambda={config.arrival_rate}, horizon={config.horizon} -> {len(arrival_times)} chegadas\n")
previous_time = 0.0
for index, arrival_time in enumerate(arrival_times):
    interval = arrival_time - previous_time
    print(f"chegada {index:>2}: t={arrival_time:6.3f}  (intervalo={interval:6.3f})")
    previous_time = arrival_time

assert all(t < config.horizon for t in arrival_times), "chegada alem do horizonte!"
print("\nOK: todas as chegadas respeitam t < horizon.")

lambda=1.5, horizon=5.0 -> 16 chegadas

chegada  0: t= 0.261  (intervalo= 0.261)
chegada  1: t= 0.370  (intervalo= 0.109)
chegada  2: t= 1.072  (intervalo= 0.702)
chegada  3: t= 1.122  (intervalo= 0.050)
chegada  4: t= 1.633  (intervalo= 0.512)
chegada  5: t= 1.937  (intervalo= 0.303)
chegada  6: t= 1.977  (intervalo= 0.040)
chegada  7: t= 2.449  (intervalo= 0.472)
chegada  8: t= 2.474  (intervalo= 0.025)
chegada  9: t= 2.853  (intervalo= 0.379)
chegada 10: t= 2.902  (intervalo= 0.048)
chegada 11: t= 2.965  (intervalo= 0.063)
chegada 12: t= 3.333  (intervalo= 0.368)
chegada 13: t= 4.502  (intervalo= 1.169)
chegada 14: t= 4.591  (intervalo= 0.088)
chegada 15: t= 4.759  (intervalo= 0.168)

OK: todas as chegadas respeitam t < horizon.


## 5) Simulacao curta com rastro visivel de cada evento

Agora juntamos tudo: `PoissonTrafficGenerator` gera as chegadas, `LoadBalancer` roteia, `Server` processa, e um `MetricsCollector` registra cada evento (`arrival`, `routing`, `service_started`, `service_completed`).

Para ver as etapas acontecendo, registramos um `event_handler` que **imprime a linha assim que o evento ocorre**, na ordem exata do relogio simulado. Como o `horizon` e pequeno (5 unidades de tempo) e a capacidade tambem (1 por servidor), a saida fica curta o suficiente para voce ler e validar manualmente.

In [21]:
def make_traced_handle(server, collector):
    """Envolve Server.handle para tambem notificar o MetricsCollector.

    O Server, por si so, so guarda seu proprio historico (`state_history`).
    Para also alimentar o MetricsCollector com `service_started`/
    `service_completed`, precisamos interceptar essas duas transicoes.
    """
    def traced_handle(request):
        with server._resource.request() as slot:  # noqa: SLF001
            server._record_state("request_received", request.id)  # noqa: SLF001
            yield slot
            request.mark_service_started(server.environment.now)
            server._record_state("service_started", request.id)  # noqa: SLF001
            collector.record_service_started(request, server)
            yield server.environment.timeout(server.service_time)
            request.mark_completed(server.environment.now)

        server._completed_count += 1  # noqa: SLF001
        server._record_state("service_completed", request.id)  # noqa: SLF001
        collector.record_service_completed(request, server)

    return traced_handle

In [22]:
trace_env = simpy.Environment()
trace_servers = [
    Server(
        trace_env,
        server_id=index,
        capacity=config.server_capacity,
        service_time=config.service_time,
    )
    for index in range(config.server_count)
]

print(f"{'tempo':>7} | {'evento':<18} | {'req':>3} | {'srv':>3} | {'ativos':>6} | {'fila':>4}")
print("-" * 60)

def print_event(event) -> None:
    server_text = "-" if event.server_id is None else str(event.server_id)
    active_text = "-" if event.active_count is None else str(event.active_count)
    waiting_text = "-" if event.waiting_count is None else str(event.waiting_count)
    print(
        f"{event.time:7.3f} | {event.event:<18} | {event.request_id:>3} | "
        f"{server_text:>3} | {active_text:>6} | {waiting_text:>4}"
    )

trace_collector = MetricsCollector(trace_env, event_handler=print_event)

for server in trace_servers:
    server.handle = make_traced_handle(server, trace_collector)  # type: ignore[assignment]

trace_load_balancer = LoadBalancer(
    trace_env,
    trace_servers,
    policy=config.policy,
    simulation_config=config,
)

def on_arrival(request) -> None:
    trace_collector.record_arrival(request)
    selected_server = trace_load_balancer.route_request(request)
    trace_collector.record_routing(request, selected_server)

traffic = PoissonTrafficGenerator(
    environment=trace_env,
    arrival_rate=config.arrival_rate,
    horizon=config.horizon,
    seed=config.seed,
)

trace_env.process(traffic.run(on_arrival=on_arrival))
trace_env.run(until=config.horizon)

  tempo | evento             | req | srv | ativos | fila
------------------------------------------------------------
  0.261 | arrival            |   0 |   - |      - |    -
  0.261 | routing            |   0 |   1 |      0 |    0
  0.261 | service_started    |   0 |   1 |      1 |    0
  0.370 | arrival            |   1 |   - |      - |    -
  0.370 | routing            |   1 |   0 |      0 |    0
  0.370 | service_started    |   1 |   0 |      1 |    0
  1.072 | arrival            |   2 |   - |      - |    -
  1.072 | routing            |   2 |   1 |      1 |    0
  1.122 | arrival            |   3 |   - |      - |    -
  1.122 | routing            |   3 |   2 |      0 |    0
  1.122 | service_started    |   3 |   2 |      1 |    0
  1.261 | service_completed  |   0 |   1 |      0 |    1
  1.261 | service_started    |   2 |   1 |      1 |    0
  1.370 | service_completed  |   1 |   0 |      0 |    0
  1.633 | arrival            |   4 |   - |      - |    -
  1.633 | routing          

### Como ler a tabela acima

- `arrival`: a requisicao chegou ao sistema (ainda sem servidor associado, por isso `srv=-`);
- `routing`: o `LoadBalancer` decidiu o servidor; nesse instante `ativos`/`fila` refletem o estado do servidor **apos** a decisao;
- `service_started`: o servidor obteve um slot livre e comecou a processar;
- `service_completed`: o servico terminou e o slot foi liberado.

Validacoes visuais rapidas que voce pode fazer:
1. Cada `request_id` deve aparecer exatamente nessa ordem: `arrival -> routing -> service_started -> service_completed`.
2. `ativos` nunca deve ultrapassar `server_capacity` (aqui, 2).
3. Se `ativos == capacity` no momento do `routing`, a proxima requisicao para o mesmo servidor deve entrar em `fila` antes de `service_started`.
4. O tempo entre `service_started` e `service_completed` do mesmo servidor deve ser sempre igual a `service_time` (aqui, 1.0).

## 6) Verificacao automatica das invariantes

Alem da leitura visual, o projeto ja possui `validate_simulation_invariants`, que confere formalmente:

- topologia e capacidade dos servidores batem com a configuracao;
- a sequencia de eventos de cada requisicao segue `arrival -> routing -> service_started -> service_completed`;
- nenhuma chegada ocorre no horizonte ou depois dele;
- conservacao: `chegadas = concluidas + ativas + aguardando`.

Rodamos essa validacao sobre a mesma execucao rastreada acima. Se nada for levantado, a simulacao esta consistente.

In [23]:
try:
    validate_simulation_invariants(config, trace_servers, trace_collector)
except SimulationInvariantError as exc:
    print(f"FALHOU: {exc}")
else:
    print("OK: nenhuma invariante violada nesta execucao.")

run_metrics = trace_collector.calculate_run_metrics(horizon=config.horizon)
print(run_metrics)

OK: nenhuma invariante violada nesta execucao.
RunMetrics(horizon=5.0, arrival_count=16, completed_count=10, pending_count=6, throughput=2.0, average_queue_time=0.4132510216437722, average_response_time=1.2483753693187347)


## Proximo passo

Se a tabela de eventos fez sentido e a validacao de invariantes imprimiu `OK`, voce pode confiar no mesmo padrao de simulacao usado em [experiments.ipynb](./experiments.ipynb), que apenas repete este processo em escala maior (mais lambdas, mais politicas, mais repeticoes) e resume os resultados via `SimulationLogger`.

Dicas para depurar mais:
- reduza `horizon`/`arrival_rate`/`server_capacity` para gerar poucas linhas e ler tudo manualmente;
- troque `policy` e rode a celula 8 novamente para comparar a distribuicao de `routing` entre servidores;
- aumente `server_capacity` para 15 (valor padrao do enunciado) e observe que as filas praticamente somem para os lambdas testados.